[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]
(https://colab.research.google.com/github/anandrao1962/ALLM-2025-FE-simple-local-rag/blob/exam-2025-final/exam/FE_2025_Final.ipynb)



# 📘 FE_2025_Final — Applications of Large Language Models (Final Exam)

**Format:** 3 hours • Open book/internet • Any LLM/agent allowed (with brief reflection)  
**Submission:** Export this notebook (`.ipynb`), a `.pdf` copy, and the two filled CSVs from Part A & Part B.

---

## What you will do
- **Part A (50 pts):** Improve *prompting/decoding* and *evaluate* vs. baseline.
- **Part B (50 pts):** Modify *retrieval* (choose one), then *evaluate* vs. baseline.
- **Reflections:** Short note on any AI/tools used in each part.
- **(Optional) +10 Bonus:** Small agentic twist (self-ask / plan-and-answer / tool-selector).

> Dataset and queries are expected at `exam/dataset/renewable_energy_exam.txt` and `exam/dataset/queries.json` in the repo.  
> If not found, a fallback demo dataset & queries are used so you can proceed.


## 🔧 Setup

In [ ]:

# Lightweight setup: avoid upgrading big preinstalled packages (torch/numpy/pandas).
# Use PyTorch-only stack; prevent TensorFlow/Flax imports to keep RAM low.
%pip -q install -U sentence-transformers==3.0.1 faiss-cpu==1.8.0 transformers==4.43.3

import os, random
import numpy as np
import torch

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

random.seed(42); np.random.seed(42); torch.manual_seed(42)
print("PyTorch:", torch.__version__)


## 📂 Load Dataset & Queries

In [ ]:

DATA_DIR = Path("exam/dataset")  # expected path in your repo
TEXT_PATH = DATA_DIR / "renewable_energy_exam.txt"
QUERIES_PATH = DATA_DIR / "queries.json"

# Fallback content (used only if files not found)
FALLBACK_TEXT = """
Early Developments
Renewable energy use dates back thousands of years. Ancient civilizations used wind to sail ships and water to grind grain. 
The first modern wind turbine for electricity was built in 1887 in Scotland.

Solar Power
Solar panels convert sunlight into electricity using photovoltaic cells. Widespread adoption began in the 1970s energy crisis.
Today, costs have dropped by over 80% in the last two decades.

Wind Power
Modern wind farms consist of multiple turbines connected to the grid. Offshore wind has grown rapidly, especially in Europe and China.
One challenge is variability: wind is not always available.

Hydropower
Hydroelectric dams generate electricity by moving water through turbines. They provide stable, large-scale power but can disrupt ecosystems and displace communities.

Environmental Impact
Renewable energy reduces greenhouse gas emissions compared to fossil fuels. However, solar panel production requires rare earth minerals, and wind farms can affect bird populations.

Future Trends
Emerging technologies include energy storage (batteries, hydrogen) and smart grids. Integration of renewables with AI-driven forecasting improves efficiency and stability.
"""

FALLBACK_QUERIES = [
    "Which country built the first modern wind turbine for electricity, and in what year?",
    "What are two environmental challenges associated with renewable energy technologies described in the text?",
    "If a region faces frequent power outages due to variability in wind, which future trend mentioned in the text could help, and why?",
    "Summarize the major advantages and drawbacks of solar, wind, and hydropower in 3–4 sentences.",
    "Suppose a government wants to balance ecological protection with renewable adoption. Which technology may present the highest social/environmental trade-offs, and why?"
]

if TEXT_PATH.exists():
    text = TEXT_PATH.read_text(encoding="utf-8")
    print(f"Loaded dataset from: {TEXT_PATH}")
else:
    text = FALLBACK_TEXT
    print("Dataset not found. Using fallback text embedded in the notebook.")

if QUERIES_PATH.exists():
    queries = json.loads(QUERIES_PATH.read_text(encoding="utf-8"))
    print(f"Loaded queries from: {QUERIES_PATH}")
else:
    queries = FALLBACK_QUERIES
    print("Queries file not found. Using fallback queries embedded in the notebook.")

print("\nSample of dataset:")
print(text.splitlines()[0:8])
print("\nQueries:", queries)


## 🧩 Chunk, Embed, and Build FAISS Index

In [ ]:

def retrieve(query, k=3):
    q = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(q, k)
    return [(int(i), float(d), chunks[int(i)]) for i, d in zip(I[0], D[0])]


In [ ]:

def retrieve(query, k=3):
    q = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(q, k)
    results = [(int(i), float(d), chunks[int(i)]) for i, d in zip(I[0], D[0])]
    return results

# Quick smoke test
for q in queries[:2]:
    hits = retrieve(q, k=3)
    print("\nQuery:", q)
    for i,(idx,score,txt) in enumerate(hits):
        print(f"  {i+1}) idx={idx} score={score:.3f} :: {txt[:80].replace('\n',' ')}...")


## ✍️ Baseline Generation (FLAN-T5) with Optional API

In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

USE_API = False  # set True to use API model instead of local generator (lowest RAM)
OPENAI_MODEL = "gpt-4o-mini"

if not USE_API:
    tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
    gen = AutoModelForSeq2SeqLM.from_pretrained(
        "google/flan-t5-small",
        low_cpu_mem_usage=True
    )

def build_prompt(query, retrieved_chunks, format_hint="Answer concisely and cite evidence from the provided context."):
    context = "\n\n".join([f"[Chunk {i}]\n{c[:1000]}" for i,(_,_,c) in enumerate(retrieved_chunks, start=1)])
    prompt = f"""You are an expert energy policy analyst.
Using only the CONTEXT below, answer the QUESTION.
If the answer isn't in the context, say "Insufficient information."

CONTEXT:
{context}

QUESTION:
{query}

RESPONSE REQUIREMENTS:
- {format_hint}
- Be faithful to the context.
"""
    return prompt

def generate_flan(prompt, max_new_tokens=128, temperature=0.2, top_k=1, top_p=1.0):
    inputs = tok(prompt, return_tensors="pt", truncation=True)
    outputs = gen.generate(
        **inputs,
        do_sample=(temperature is not None and temperature > 0),
        temperature=temperature,
        top_k=top_k, top_p=top_p,
        max_new_tokens=max_new_tokens
    )
    return tok.decode(outputs[0], skip_special_tokens=True)

def generate_openai(prompt):
    import os
    try:
        from openai import OpenAI
        client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role":"user","content":prompt}],
            temperature=0.2
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"[OpenAI Error] {e}"

def answer_query_baseline(query, k=3, format_hint="Provide a short answer."):
    retrieved = retrieve(query, k=k)
    prompt = build_prompt(query, retrieved, format_hint=format_hint)
    if USE_API:
        return generate_openai(prompt), prompt, retrieved
    else:
        return generate_flan(prompt), prompt, retrieved



# 🅐 Part A — Prompting & Decoding (50 pts)

**Your tasks (do both):**
1) Make **two prompt modifications** (e.g., persona, structured output, add reasoning instruction).  
2) Try **two decoding strategies** (e.g., adjust temperature/top-k/top-p).  

Then compare Baseline vs. Modified on all 5 queries using the 1–4 scoring for **Correctness** and **Coherence**.


In [ ]:

# ==== TODO: Prompt Modifications (Edit below) ====
FORMAT_HINT = "Answer in 3 bullet points, each <20 words, and cite the chunk number(s)."
PERSONA_PREFIX = "You are an expert energy policy analyst. Think step-by-step and be faithful to the context."

def build_prompt_modified(query, retrieved_chunks):
    context = "\n\n".join([f"[Chunk {i}]\n{c[:1000]}" for i,(_,_,c) in enumerate(retrieved_chunks, start=1)])
    prompt = f"""{PERSONA_PREFIX}
Using only the CONTEXT below, answer the QUESTION.
If the answer isn't in the context, say "Insufficient information."

CONTEXT:
{context}

QUESTION:
{query}

RESPONSE REQUIREMENTS:
- {FORMAT_HINT}
"""
    return prompt


In [ ]:

# ==== TODO: Decoding Strategies (Edit these for experiments) ====
# Strategy A (e.g., top-k sampling)
DECODING_A = dict(max_new_tokens=160, temperature=0.7, top_k=20, top_p=1.0)

# Strategy B (e.g., more deterministic)
DECODING_B = dict(max_new_tokens=160, temperature=0.2, top_k=1, top_p=1.0)

def answer_query_modified(query, k=3, decoding="A"):
    retrieved = retrieve(query, k=k)
    prompt = build_prompt_modified(query, retrieved)
    params = DECODING_A if decoding=="A" else DECODING_B
    if USE_API:
        # API path ignores fine-grained decoding here (keep it simple)
        out = generate_openai(prompt)
    else:
        out = generate_flan(prompt, **params)
    return out, prompt, retrieved


In [ ]:

# Run baseline and modified for all queries and collect outputs
def run_partA_experiment(queries, k=3):
    rows = []
    outputs = {"baseline": [], "modified_A": [], "modified_B": []}
    for qi, q in enumerate(queries, start=1):
        base_out, base_p, base_ret = answer_query_baseline(q, k=k)
        modA_out, modA_p, modA_ret = answer_query_modified(q, k=k, decoding="A")
        modB_out, modB_p, modB_ret = answer_query_modified(q, k=k, decoding="B")
        outputs["baseline"].append(base_out)
        outputs["modified_A"].append(modA_out)
        outputs["modified_B"].append(modB_out)
        rows.append(dict(query_id=qi, query=q))
    return pd.DataFrame(rows), outputs

partA_df, partA_outputs = run_partA_experiment(queries, k=3)
partA_df



### 🧮 Fill the Part A Evaluation Table (1–4 scale)
- 1 = Poor, 2 = Weak, 3 = Good, 4 = Excellent  
- Compare baseline vs. modified outputs for each query. Add short notes.


In [ ]:

# Create empty Part A evaluation table
partA_eval = pd.DataFrame({
    "Query #": list(range(1, len(queries)+1)),
    "Baseline Correctness (1-4)": [None]*len(queries),
    "Baseline Coherence (1-4)": [None]*len(queries),
    "Modified-A Correctness (1-4)": [None]*len(queries),
    "Modified-A Coherence (1-4)": [None]*len(queries),
    "Modified-B Correctness (1-4)": [None]*len(queries),
    "Modified-B Coherence (1-4)": [None]*len(queries),
    "Notes": [""]*len(queries)
})
partA_eval


In [ ]:

# Helper: show outputs side-by-side for quick judging
def show_outputs_for_query(qi):
    i = qi - 1
    print("Query:", queries[i], "\n")
    print("BASELINE:\n", partA_outputs["baseline"][i], "\n")
    print("MODIFIED A:\n", partA_outputs["modified_A"][i], "\n")
    print("MODIFIED B:\n", partA_outputs["modified_B"][i], "\n")

# Example: show query 1 outputs
show_outputs_for_query(1)


In [ ]:

# Save Part A evaluation to CSV
Path("exam/templates").mkdir(parents=True, exist_ok=True)
partA_path = Path("exam/templates/partA_eval_template.csv")
partA_eval.to_csv(partA_path, index=False)
print("Saved:", partA_path.resolve())



### 🖊️ Part A Reflection (3–4 sentences)
Briefly note which AI tools/LLMs (if any) you used to help with prompts/decoding and how reliable they were.



# 🅑 Part B — Retrieval & Evaluation (50 pts)

**Choose ONE modification:**
- Change **chunk size/overlap**
- Swap **embedding model** (will re-index)
- Add a simple **re-ranking** rule (e.g., keyword overlap)

Then compare **Precision, Recall, and Answer Relevance (1–4)** vs. baseline for all 5 queries.


In [ ]:

# ==== Option 1: Change chunking ====
def rebuild_with_chunking(new_chunk_size=200, new_overlap=50):
    global CHUNK_SIZE, OVERLAP, chunks, index, chunk_embs
    CHUNK_SIZE, OVERLAP = new_chunk_size, new_overlap
    chunks = chunk_text(text, CHUNK_SIZE, OVERLAP)
    print(f"Rebuilt chunks: {len(chunks)} with size={CHUNK_SIZE}, overlap={OVERLAP}")
    # Re-embed & re-index
    global embed_model
    index, chunk_embs = build_faiss(chunks, embed_model)
    print("Rebuilt FAISS index with existing embedding model:", EMBED_MODEL_NAME)

# ==== Option 2: Swap embedding model ====
def rebuild_with_embeddings(new_model_name="sentence-transformers/all-mpnet-base-v2"):
    global EMBED_MODEL_NAME, embed_model, index, chunk_embs
    EMBED_MODEL_NAME = new_model_name
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    index, chunk_embs = build_faiss(chunks, embed_model)
    print("Rebuilt FAISS index with new embedding model:", EMBED_MODEL_NAME)

# ==== Option 3: Simple re-ranking rule (keyword overlap) ====
import re
def keyword_overlap_score(query, text):
    q_tokens = set(re.findall(r"[A-Za-z0-9]+", query.lower()))
    t_tokens = set(re.findall(r"[A-Za-z0-9]+", text.lower()))
    if not q_tokens:
        return 0
    return len(q_tokens & t_tokens) / len(q_tokens)

def retrieve_with_rerank(query, k=3, pool=6):
    # retrieve a larger pool, then rerank by overlap
    base = retrieve(query, k=pool)
    rescored = sorted(base, key=lambda x: keyword_overlap_score(query, x[2]), reverse=True)
    return rescored[:k]


In [ ]:

# Helper to generate an answer with either base retrieve() or reranked
def answer_query_with_retrieval(query, use_rerank=False, k=3, decoding="B"):
    if use_rerank:
        retrieved = retrieve_with_rerank(query, k=k, pool=max(6, 2*k))
    else:
        retrieved = retrieve(query, k=k)
    prompt = build_prompt_modified(query, retrieved)
    params = DECODING_B  # a steady decoding choice for Part B
    if USE_API:
        out = generate_openai(prompt)
    else:
        out = generate_flan(prompt, **params)
    return out, retrieved



### 🧮 Fill the Part B Evaluation Table
- **Context Precision (%):** % of retrieved chunks that are relevant to the answer.  
- **Context Recall (%):** % of all relevant chunks (in dataset) retrieved.  
- **Answer Relevance (1–4):** How well the final answer addresses the query.
> You may estimate precision/recall manually given the small dataset. Add notes if you approximate.


In [ ]:

# Build an empty Part B evaluation table
partB_cols = [
    "Query #",
    "Baseline Precision (%)", "Baseline Recall (%)", "Baseline Relevance (1-4)",
    "Modified Precision (%)", "Modified Recall (%)", "Modified Relevance (1-4)",
    "Notes"
]
partB_eval = pd.DataFrame([{ "Query #": i+1 } for i in range(len(queries))], columns=partB_cols)
partB_eval


In [ ]:

# Example runner to help students populate their tables (students still judge relevance)
def run_partB_compare(queries, use_rerank=False):
    results = []
    for qi, q in enumerate(queries, start=1):
        # baseline
        base_ans, base_ret = answer_query_with_retrieval(q, use_rerank=False, k=3)
        # modified
        mod_ans, mod_ret = answer_query_with_retrieval(q, use_rerank=use_rerank, k=3)
        results.append(dict(
            query_id=qi, query=q,
            baseline_answer=base_ans, baseline_chunks=[r[2] for r in base_ret],
            modified_answer=mod_ans, modified_chunks=[r[2] for r in mod_ret]
        ))
    return results

# Example: if you picked re-ranking as your modification, set use_rerank=True
example_results = run_partB_compare(queries, use_rerank=False)
len(example_results), example_results[0].keys()


In [ ]:

# Save Part B evaluation to CSV
partB_path = Path("exam/templates/partB_eval_template.csv")
partB_eval.to_csv(partB_path, index=False)
print("Saved:", partB_path.resolve())



### 🖊️ Part B Reflection (3–4 sentences)
Briefly note which AI tools/LLMs (if any) you used to help with retrieval changes and how reliable they were.



---
## ⚡ Optional Bonus (+10 pts): Agentic Twist
Choose **one**:
1) **Self-Ask**: Generate a follow-up sub-question, retrieve, then answer.  
2) **Plan-and-Answer**: Create a mini-plan (e.g., solar→wind→hydro), retrieve per step, then synthesize.  
3) **Tool-Selector**: If fact-based (who/when/where) → short answer; if synthesis → longer structured answer.


In [ ]:

def self_ask(query):
    # very simple heuristic "self-ask" followed by retrieval and answering
    follow_up = f"What specific facts are needed to answer: '{query}'?"
    _, follow_prompt, _ = answer_query_baseline(follow_up, k=2, format_hint="List 1-2 facts to look up.")
    # Now retrieve/answer original
    ans, _, _ = answer_query_baseline(query, k=3, format_hint="Short, evidence-based answer with chunk refs.")
    return {"follow_up": follow_up, "follow_prompt": follow_prompt, "answer": ans}

def plan_and_answer(query):
    plan = ["Find solar info", "Find wind info", "Find hydropower info"]
    gathered = []
    for step in plan:
        step_q = f"{step} relevant to: {query}"
        ans, _, _ = answer_query_baseline(step_q, k=2, format_hint="Extract key facts only.")
        gathered.append(ans)
    # Final synthesis
    synth_q = f"Synthesize the best answer to: {query} using facts above."
    final, _, _ = answer_query_baseline(synth_q, k=3, format_hint="3 concise bullet points with chunk refs.")
    return {"plan": plan, "gathered": gathered, "final": final}

def tool_selector(query):
    fact_words = ["who","when","where","which","what year"]
    is_facty = any(w in query.lower() for w in fact_words)
    fmt = "One-sentence answer with a chunk reference." if is_facty else "3 bullet points, concise, with chunk refs."
    ans, _, _ = answer_query_baseline(query, k=3, format_hint=fmt)
    return {"mode": "FACT" if is_facty else "SYNTH", "answer": ans}

# Demo (you can change the query id)
demo = plan_and_answer(queries[0])
demo



---

## 💾 Save/Export Helpers
Use these to ensure your CSVs are saved and ready to download alongside your notebook.


In [ ]:

# Re-save both CSVs (in case you've edited the DataFrames)
partA_eval.to_csv("exam/templates/partA_eval_template.csv", index=False)
partB_eval.to_csv("exam/templates/partB_eval_template.csv", index=False)
print("Saved: exam/templates/partA_eval_template.csv")
print("Saved: exam/templates/partB_eval_template.csv")



---

## ✅ Submission Checklist
- [ ] Ran setup and index build
- [ ] Completed **Part A**: two prompt changes + two decoding strategies; filled Part A table
- [ ] Completed **Part B**: one retrieval change; filled Part B table
- [ ] Wrote both **Reflections**
- [ ] (Optional) Agentic Bonus
- [ ] Exported **.ipynb**, **.pdf**, and the two **CSV** files
